# 05 · Evaluation

The ground-truth labels are opened here and nowhere else in the pipeline, the
way a held-out test set would be.

Three questions: how good is the rule layer, which rules earn their alerts, and
how much does combining rules with the model reduce the analyst's workload.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import warnings; warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import config

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
                     "axes.edgecolor": "#e3e2dc", "grid.color": "#e3e2dc",
                     "axes.titleweight": "bold", "figure.dpi": 110,
                     "axes.spines.top": False, "axes.spines.right": False})
S1, S2, S3 = "#2a78d6", "#eb6834", "#1baf7a"

In [2]:
import json, rules
from ml_layer import fit_score

df = pd.read_csv(config.CLEAN_TRANSACTIONS, parse_dates=["timestamp"])
truth = (pd.read_csv(config.GROUND_TRUTH).drop_duplicates("transaction_id")
           .set_index("transaction_id")["is_laundering_pattern"])
y = df.transaction_id.map(truth).fillna(0).astype(int)
fired = rules.apply_all(df)
scores, _, _ = fit_score(df)
print(f"{len(df):,} transactions, {int(y.sum())} known cases ({y.mean():.2%} prevalence)")

60,059 transactions, 139 known cases (0.23% prevalence)


## The rule layer, rule by rule

In [3]:
def score_set(flag):
    tp = int((flag & (y == 1)).sum()); n = int(flag.sum())
    return {"alerts": n, "caught": tp, "false positives": n - tp,
            "precision": f"{tp/n:.2%}" if n else "-",
            "recall": f"{tp/y.sum():.1%}"}

per_rule = pd.DataFrame({r: score_set(fired[r]) for r in rules.RULES}).T
per_rule

,alerts,caught,false positives,precision,recall
structuring,100,100,0,100.00%,71.9%
rapid_movement,80,40,40,50.00%,28.8%
high_risk_country,5178,13,5165,0.25%,9.4%
round_amount,49,2,47,4.08%,1.4%


Judged on recall alone, `high_risk_country` looks like a contributor: it caught
13 cases. The question that matters is different.

## Which rules earn their alerts?

A rule that only ever re-finds cases another rule already caught is pure volume,
whatever its own recall says.

In [4]:
unique = {}
for r in rules.RULES:
    others = fired[[o for o in rules.RULES if o != r]].any(axis=1)
    only = fired[r] & ~others
    unique[r] = {"alerts only this rule raises": int(only.sum()),
                 "cases found by no other rule": int((only & (y == 1)).sum())}
pd.DataFrame(unique).T

,alerts only this rule raises,cases found by no other rule
structuring,98,98
rapid_movement,61,25
high_risk_country,5158,0
round_amount,44,0


**This is the finding the project turns on.**

`high_risk_country` raises 5,158 alerts nobody else raises and finds **zero**
cases nobody else found. Every case it "caught" was already caught by
structuring or rapid movement. Same for `round_amount`.

96% of the alert volume comes from screens that contribute no unique
detections.

## The baseline

In [5]:
baseline = score_set(fired.rule_alert)
print(f"alerts          : {baseline['alerts']:,}")
print(f"cases caught    : {baseline['caught']}/{int(y.sum())}  ({baseline['recall']})")
print(f"false positives : {baseline['false positives']:,}")
print(f"precision       : {baseline['precision']}")

alerts          : 5,384
cases caught    : 139/139  (100.0%)
false positives : 5,245
precision       : 2.58%


100% recall, 2.6% precision. The rules find everything and bury the analyst,
which is exactly the real-world failure mode.

## Tiering

Targeted typologies escalate on their own. Broad screens escalate only when the
model also finds the transaction unusual.

In [6]:
tier_a = fired[["structuring", "rapid_movement"]].any(axis=1)
tier_b = fired[["high_risk_country", "round_amount"]].any(axis=1)

sweep = []
for pct in [.50, .75, .90, .95, .99, .999]:
    combined = tier_a | (tier_b & (scores.anomaly_pct >= pct))
    r = score_set(combined); r["threshold"] = pct
    sweep.append(r)
pd.DataFrame(sweep).set_index("threshold")

,alerts,caught,false positives,precision,recall
threshold,,,,,
0.500,2527,139,2388,5.50%,100.0%
0.750,1200,139,1061,11.58%,100.0%
0.900,538,139,399,25.84%,100.0%
0.950,362,139,223,38.40%,100.0%
0.990,219,139,80,63.47%,100.0%
0.999,181,139,42,76.80%,100.0%


Recall stays at 100% across the entire sweep, because **Tier A alone catches all
139 cases**. Tier B cannot add a true positive; it can only add volume.

Pure optimisation therefore says delete Tier B. I did not, and the reason is not
statistical: twelve months of synthetic data showing no laundering through
Cyprus is not evidence that none exists, and a jurisdiction screen carries
supervisory expectations a precision figure does not capture. The corroboration
requirement is the compromise.

## The shipped configuration

In [7]:
theta = 0.99
corroborated = tier_b & (scores.anomaly_pct >= theta)
combined = tier_a | corroborated
final = combined | ((scores.anomaly_pct >= .999) & ~combined)

comparison = pd.DataFrame({
    "rules only": score_set(fired.rule_alert),
    "tier A only": score_set(tier_a),
    "shipped (rules + anomaly)": score_set(final),
}).T
comparison

,alerts,caught,false positives,precision,recall
rules only,5384,139,5245,2.58%,100.0%
tier A only,179,139,40,77.65%,100.0%
shipped (rules + anomaly),228,139,89,60.96%,100.0%


In [8]:
b, f = score_set(fired.rule_alert), score_set(final)
print(f"alert volume    : {b['alerts']:,} -> {f['alerts']:,}")
print(f"false positives : {b['false positives']:,} -> {f['false positives']:,}"
      f"   ({100*(b['false positives']-f['false positives'])/b['false positives']:.1f}% cut)")
print(f"precision       : {b['precision']} -> {f['precision']}")
print(f"cases caught    : {b['caught']} -> {f['caught']} (of {int(y.sum())})")
print(f"\nanalyst hours saved at 15 min a review: {(b['alerts']-f['alerts'])*0.25:,.0f}/year")

alert volume    : 5,384 -> 228
false positives : 5,245 -> 89   (98.3% cut)
precision       : 2.58% -> 60.96%
cases caught    : 139 -> 139 (of 139)

analyst hours saved at 15 min a review: 1,289/year


## Under adversarial pressure

`src/adversarial.py` perturbs only the laundering transactions and re-runs the
whole stack with the model frozen. Results loaded here.

In [9]:
ev = json.loads((config.OUTPUTS / "adversarial.json").read_text())
total = ev[0]["known_cases"]
pd.DataFrame([{ "evasion": e["scenario"],
                "rules": round(e["rules_recall"] * total),
                "shipped": round(e["shipped_recall"] * total),
                "model alone": round(e["ml_alone_recall"] * total)} for e in ev])

,evasion,rules,shipped,model alone
0,No evasion (control),139,139,120
1,Structuring: smaller payments,41,63,91
2,Structuring: slower burst,41,47,63
3,Rapid movement: wait 30 hours,114,130,119
4,Rapid movement: keep 30% back,114,128,115
5,Round amounts: add pence,139,139,119
6,All of the above,14,39,44


The rule layer collapses from 139 to 14 against an informed launderer. The model
degrades instead of collapsing, and beats the rules in every scenario.

That inverts the conclusion above: on static data the rules detect and the model
filters noise, but under adaptation the roles swap. It also exposes the
architecture as the weak link, since the shipped system catches fewer than the
model alone at the same budget when everything is evaded at once.

## The analyst queue

The deliverable an analyst would actually work from: every alert carrying the
reason it surfaced.

In [10]:
cases = pd.read_csv(config.CASE_MANAGEMENT)
print(f"{len(cases)} alerts")
print(cases.escalation_channel.value_counts().to_string())
cases.head(5)[["transaction_id", "amount", "reason", "anomaly_pct", "priority"]]

228 alerts
escalation_channel
rule: targeted typology                       179
rule: broad screen + anomaly corroboration     40
anomaly only                                    9


,transaction_id,amount,reason,anomaly_pct,priority
0,TXNS00923,9310.61,structuring,1.0000,high
1,TXNR0162,27183.65,rapid_movement,1.0000,high
2,TXNS00330,9386.00,"structuring, round_amount",1.0000,high
3,TXNR0231,42106.83,rapid_movement,1.0000,high
4,TXNS01120,9878.75,structuring,0.9999,high
